In [1]:
import os

FOLDER = "swimdata/"

files = os.listdir(FOLDER)

In [2]:
first = files[0]
print(first)
name, age, _, _ = first.removesuffix(".txt").split("-")
name, age

Abi-10-100m-Back.txt


('Abi', '10')

In [3]:
import DBcm

db_details = "CoachDB.sqlite3"

In [4]:
with DBcm.UseDatabase(db_details) as db:
    SQL_0 = """
    create table if not exists swimmers (
        id integer not null primary key autoincrement,
        name varchar(32) not null,
        age integer not null
    )
    """

    SQL_1 = """
        create table if not exists events (
            id integer not null primary key autoincrement,
            distance varchar(16) not null,
            stroke varchar(16) not null
        )
    """
    SQL_2 = """
        create table if not exists times (
            swimmer_id integer not null,
            event_id integer not null,
            time varchar(16) not null,
            ts timestamp default current_timestamp
        )
    """
    db.execute(SQL_0)
    db.execute(SQL_1)
    db.execute(SQL_2)

In [5]:
SQL_INSERT = """
    insert into swimmers
    (name, age)
    values
    (?, ?)
"""
print(SQL_INSERT)


    insert into swimmers
    (name, age)
    values
    (?, ?)



In [6]:
with DBcm.UseDatabase(db_details) as db:
    db.execute("pragma table_list")
    results = db.fetchall()
results

[('main', 'times', 'table', 4, 0, 0),
 ('main', 'events', 'table', 3, 0, 0),
 ('main', 'sqlite_sequence', 'table', 2, 0, 0),
 ('main', 'swimmers', 'table', 3, 0, 0),
 ('main', 'sqlite_schema', 'table', 5, 0, 0),
 ('temp', 'sqlite_temp_schema', 'table', 5, 0, 0)]

In [7]:
SQL = """select * from swimmers"""
with DBcm.UseDatabase(db_details) as db:
    db.execute(SQL)
    results = db.fetchall()
results

[]

In [8]:
SQL = """delete from swimmers"""
with DBcm.UseDatabase(db_details) as db:
    db.execute(SQL)

In [9]:
db_details = "CoachDB.sqlite3"

FOLDER = "swimdata/"

files = os.listdir(FOLDER)

SQL_SELECT = """
    select * from swimmers
    where name = ? and age = ?
"""

SQL_INSERT = """
    insert into swimmers
    (name, age)
    values
    (?, ?)
"""

with DBcm.UseDatabase(db_details) as db:
    for fn in files:
        name, age, _, _ = fn.removesuffix(".txt").split("-")
        db.execute(SQL_SELECT, (name, age,))
        if db.fetchall():
            continue
        db.execute(SQL_INSERT, (name, age,))

In [10]:
SQL_SELECT = """
    select * from swimmers
    where name = ? and age = ?
"""

SQL_INSERT = """
    insert into swimmers
    (name, age)
    values
    (?, ?)
"""

with DBcm.UseDatabase(db_details) as db:
    for fn in files:
        name, age, _, _ = fn.removesuffix(".txt").split("-")
        db.execute(SQL_SELECT, (name, age,))
        if db.fetchall():
            continue
        db.execute(SQL_INSERT, (name, age,))

In [11]:
with (DBcm.UseDatabase(db_details) as db):
    SQL = "SELECT * FROM swimmers"
    db.execute(SQL)
    results = db.fetchall()
results

[(1, 'Abi', 10),
 (2, 'Ali', 12),
 (3, 'Alison', 14),
 (4, 'Aurora', 13),
 (5, 'Bill', 18),
 (6, 'Blake', 15),
 (7, 'Calvin', 9),
 (8, 'Carl', 15),
 (9, 'Chris', 17),
 (10, 'Darius', 13),
 (11, 'Dave', 17),
 (12, 'Elba', 14),
 (13, 'Emma', 13),
 (14, 'Erika', 15),
 (15, 'Hannah', 13),
 (16, 'Katie', 9),
 (17, 'Lizzie', 14),
 (18, 'Maria', 9),
 (19, 'Mike', 15),
 (20, 'Owen', 15),
 (21, 'Ruth', 13),
 (22, 'Tasmin', 15)]

In [12]:
SQL_SELECT = """
    select * from events
    where distance = ? and stroke = ?
"""

SQL_INSERT = """
    insert into events
    (distance, stroke)
    values
    (?, ?)
"""

with DBcm.UseDatabase(db_details) as db:
    for fn in files:
        _, _, distance, stroke = fn.removesuffix(".txt").split("-")
        db.execute(SQL_SELECT, (distance, stroke,))
        if db.fetchall():
            continue
        db.execute(SQL_INSERT, (distance, stroke,))

In [13]:
with (DBcm.UseDatabase(db_details) as db):
    SQL = "SELECT * FROM events"
    db.execute(SQL)
    results = db.fetchall()
results

[(1, '100m', 'Back'),
 (2, '100m', 'Breast'),
 (3, '50m', 'Back'),
 (4, '50m', 'Breast'),
 (5, '50m', 'Free'),
 (6, '100m', 'Free'),
 (7, '200m', 'Back'),
 (8, '100m', 'Fly'),
 (9, '50m', 'Fly'),
 (10, '200m', 'IM'),
 (11, '200m', 'Breast'),
 (12, '200m', 'Free'),
 (13, '400m', 'Free')]

Populating the time table

In [14]:
SQL_GET_SWIMMER = """
select id from swimmers
where name = ? and age = ?"""

In [15]:
SQL_GET_EVENT = """
select id from events
where distance = ? and stroke = ?"""

In [16]:
SQL_INSERT = """
insert into times
(swimmer_id, event_id, time)
values
(?, ?, ?)"""

In [17]:
with DBcm.UseDatabase(db_details) as db:
    for fn in files:
        name, age, distance, stroke = fn.removesuffix(".txt").split("-")
        db.execute(SQL_GET_SWIMMER, (name, age,))
        s_id = db.fetchone()[0]
        db.execute(SQL_GET_EVENT, (distance, stroke,))
        e_id = db.fetchone()[0]
        with open(FOLDER+fn) as sf:
            times = sf.read().strip().split(",")
            for t in times:
                db.execute(SQL_INSERT, (s_id, e_id, t,))

In [18]:
SQL = """select count(*) from times"""
with DBcm.UseDatabase(db_details) as db:
    db.execute(SQL)
    results = db.fetchone()[0]
results

338

In [19]:
SQL = """select * from times limit 10"""
with DBcm.UseDatabase(db_details) as db:
    db.execute(SQL)
    results = db.fetchall()
results

[(1, 1, '1:31.59', '2026-03-29 06:40:10'),
 (1, 1, '1:26.55', '2026-03-29 06:40:10'),
 (1, 1, '1:28.75', '2026-03-29 06:40:10'),
 (1, 1, '1:39.79', '2026-03-29 06:40:10'),
 (1, 1, '1:32.37', '2026-03-29 06:40:10'),
 (1, 2, '1:42.97', '2026-03-29 06:40:10'),
 (1, 2, '1:43.31', '2026-03-29 06:40:10'),
 (1, 2, '1:43.50', '2026-03-29 06:40:10'),
 (1, 2, '1:40.34', '2026-03-29 06:40:10'),
 (1, 3, '41.50', '2026-03-29 06:40:10')]